Fuzzy clustering of both veg space and optical space

In [3]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score

# =============================================================================
# PATHS
# =============================================================================

csv_path = Path(r"C:\NCA_DATA\Vegetation Data\NCA_Master_Aligned_spp_fg.csv")
out_dir = Path(r"C:\NCA_DATA\Vegetation Data\cluster\veg_cluster_discrete")
out_dir.mkdir(parents=True, exist_ok=True)

sil_plot_png = out_dir / "veg_silhouette.png"
wcss_plot_png = out_dir / "veg_wcss.png"
ch_plot_png = out_dir / "veg_calinski_harabasz.png"

# =============================================================================
# SETTINGS
# =============================================================================

K_MIN = 2
K_MAX = 25
RANDOM_STATE = 42
N_INIT = 20
MAX_ITER = 1000

SIL_SAMPLE_SIZE = 10000   # set None to use all rows
USE_MINIBATCH = False

# exploratory only; change later after inspecting diagnostics
FINAL_K = 16

# =============================================================================
# LOAD DATA
# =============================================================================

print("Reading vegetation dataset...")
df = pd.read_csv(csv_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

# =============================================================================
# IDENTIFY COLUMNS
# =============================================================================

meta_cols = {
    "Plot", "Year",
    "total_cover", "total_cover_fg",
    "SHRUB_total"
}
fg_cols = [c for c in df.columns if c.endswith("_FG")]
species_cols = [c for c in df.columns if c not in meta_cols and c not in fg_cols]

print(f"Detected {len(species_cols)} species/catchall columns.")
print(f"Detected {len(fg_cols)} FG columns.")

# =============================================================================
# BUILD MATRIX
# =============================================================================

X_raw = df[species_cols].copy()

for c in species_cols:
    X_raw[c] = pd.to_numeric(X_raw[c], errors="coerce")

X_raw = X_raw.fillna(0.0)

row_sums = X_raw.sum(axis=1).values
keep = row_sums > 0

if not np.all(keep):
    print(f"Dropping {(~keep).sum():,} rows with zero species total.")

df_work = df.loc[keep].copy().reset_index(drop=True)
X_raw = X_raw.loc[keep].reset_index(drop=True)

# Hellinger transform
print("Applying Hellinger transform...")
row_sums = X_raw.sum(axis=1).values[:, None]
X_prop = X_raw.values / row_sums
X = np.sqrt(X_prop)

print(f"Matrix shape: {X.shape}")

# =============================================================================
# HELPERS
# =============================================================================

def fit_kmeans(X: np.ndarray, k: int, random_state: int):
    if USE_MINIBATCH:
        km = MiniBatchKMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=N_INIT,
            max_iter=MAX_ITER,
            batch_size=8192
        )
    else:
        km = KMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=N_INIT,
            max_iter=MAX_ITER
        )
    labels = km.fit_predict(X)
    return km, labels

def save_line_plot(x, y, xlabel, ylabel, title, out_path):
    plt.figure(figsize=(8, 5))
    plt.plot(x, y, marker="o")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

# =============================================================================
# K DIAGNOSTICS
# =============================================================================

print("Running K diagnostics...")
results = []

n = X.shape[0]
if SIL_SAMPLE_SIZE is not None and n > SIL_SAMPLE_SIZE:
    rng = np.random.default_rng(RANDOM_STATE)
    sil_idx = rng.choice(n, size=SIL_SAMPLE_SIZE, replace=False)
    X_sil = X[sil_idx]
else:
    sil_idx = None
    X_sil = X

for k in range(K_MIN, K_MAX + 1):
    print(f"  k = {k}")
    km, labels = fit_kmeans(X, k, RANDOM_STATE)

    if sil_idx is not None:
        sil = silhouette_score(X_sil, labels[sil_idx], metric="euclidean")
    else:
        sil = silhouette_score(X, labels, metric="euclidean")

    ch = calinski_harabasz_score(X, labels)

    results.append({
        "k": k,
        "wcss": float(km.inertia_),
        "silhouette": float(sil),
        "calinski_harabasz": float(ch)
    })

metrics = pd.DataFrame(results)

# top performers
top_sil = metrics.sort_values("silhouette", ascending=False).head(5)
top_ch = metrics.sort_values("calinski_harabasz", ascending=False).head(5)

print("\nTop K by silhouette:")
print(top_sil)

print("\nTop K by Calinski-Harabasz:")
print(top_ch)

# union of candidates
candidate_k = sorted(set(top_sil["k"]).union(set(top_ch["k"])))
print("\nCandidate K values:", candidate_k)

# =============================================================================
# PLOTS
# =============================================================================

save_line_plot(
    metrics["k"], metrics["silhouette"],
    "k", "Mean silhouette",
    "Vegetation clustering: silhouette",
    sil_plot_png
)

save_line_plot(
    metrics["k"], metrics["wcss"],
    "k", "WCSS",
    "Vegetation clustering: WCSS",
    wcss_plot_png
)

save_line_plot(
    metrics["k"], metrics["calinski_harabasz"],
    "k", "Calinski-Harabasz score",
    "Vegetation clustering: Calinski-Harabasz",
    ch_plot_png
)

print("\nPlots written:")
print(sil_plot_png)
print(wcss_plot_png)
print(ch_plot_png)

# =============================================================================
# CONSOLE OUTPUT
# =============================================================================

print("\nDiagnostics:")
print(metrics.to_string(index=False))

print("\nTop k by silhouette:")
print(metrics.sort_values("silhouette", ascending=False).head(10).to_string(index=False))

print("\nTop k by Calinski-Harabasz:")
print(metrics.sort_values("calinski_harabasz", ascending=False).head(10).to_string(index=False))

# =============================================================================
# OPTIONAL FINAL FIT FOR QUICK LOOK
# =============================================================================

print(f"\nQuick exploratory fit at k = {FINAL_K}...")
km_final, final_labels = fit_kmeans(X, FINAL_K, RANDOM_STATE)

df_work["veg_cluster"] = final_labels + 1

print("\nCluster sizes:")
print(df_work["veg_cluster"].value_counts().sort_index().to_string())

species_summary = (
    df_work.groupby("veg_cluster")[species_cols]
    .mean()
)

print("\nTop 10 species/catchall means by cluster:")
for cluster_id in species_summary.index:
    top = species_summary.loc[cluster_id].sort_values(ascending=False).head(10)
    print(f"\nCluster {cluster_id}")
    print(top.to_string())

if fg_cols:
    fg_summary = (
        df_work.groupby("veg_cluster")[fg_cols]
        .mean()
    )

    print("\nFG means by cluster:")
    print(fg_summary.to_string())


for k in candidate_k:
    print(f"\n=== Inspecting k = {k} ===")

    km, labels = fit_kmeans(X, k, RANDOM_STATE)
    df_work["veg_cluster"] = labels + 1

    # cluster sizes
    sizes = df_work["veg_cluster"].value_counts().sort_index()
    print("\nCluster sizes:")
    print(sizes)

    # FG summaries (much more interpretable than species)
    fg_summary = (
        df_work.groupby("veg_cluster")[fg_cols]
        .mean()
    )

    print("\nFG summary:")
    print(fg_summary.round(2))

Reading vegetation dataset...
Rows: 717
Columns: 86
Detected 71 species/catchall columns.
Detected 10 FG columns.
Applying Hellinger transform...
Matrix shape: (717, 71)
Running K diagnostics...
  k = 2
  k = 3
  k = 4
  k = 5


c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\c

  k = 6
  k = 7
  k = 8


c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(


  k = 9
  k = 10
  k = 11


...


Cluster sizes:
veg_cluster
1     158
2      94
3      13
4      41
5      57
6      43
7      73
8      32
9      75
10     69
11     39
12     16
13      7
Name: count, dtype: int64

FG summary:
             ARTR_FG  BAREGROUND_FG  BIOCRUST_FG  EAG_FG  EF_FG  LITTER_FG  \
veg_cluster                                                                  
1               1.43           8.45         4.00   59.63   6.08       7.08   
2               2.82          29.15        13.25    3.87  11.55      14.79   
3              17.86          24.75        16.20    6.00   6.11      20.22   
4               0.00          25.93        21.24    2.68  29.22      12.76   
5               0.34          17.32         9.86   16.73   1.82      12.57   
6               1.62          12.15         4.50   61.84   5.66       6.62   
7               1.61          18.11        11.54    4.07  10.28       8.79   
8               0.08          24.44         7.41   10.56   0.90      20.04   
9               0.20   

c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(



Cluster sizes:
veg_cluster
1      28
2      30
3      50
4     139
5      41
6      66
7      28
8      46
9      66
10     60
11     16
12      9
13     86
14     29
15     23
Name: count, dtype: int64

FG summary:
             ARTR_FG  BAREGROUND_FG  BIOCRUST_FG  EAG_FG  EF_FG  LITTER_FG  \
veg_cluster                                                                  
1               0.00          63.12         2.18    1.52  15.23       6.62   
2               0.08          22.28         7.75    9.69   0.92      19.66   
3               0.16          17.43         9.80   15.91   1.05      13.01   
4               2.29           6.42         4.17   63.13   3.75       6.87   
5               0.00          25.93        21.24    2.68  29.22      12.76   
6               1.01          20.90         9.86    3.60  10.66       9.47   
7               0.01          36.84        21.31    1.26  19.00      10.55   
8               4.15          22.98         4.84   42.40   1.12       9.14   
9  

c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(



Cluster sizes:
veg_cluster
1      67
2      68
3      25
4      20
5      19
6      10
7     138
8       7
9      57
10     24
11      9
12     13
13     41
14     22
15      9
16     51
17     14
18     92
19     31
Name: count, dtype: int64

FG summary:
             ARTR_FG  BAREGROUND_FG  BIOCRUST_FG  EAG_FG  EF_FG  LITTER_FG  \
veg_cluster                                                                  
1               0.09          17.33         9.69   10.29   1.32      17.12   
2               0.98          18.80        11.34    3.25  10.86       8.40   
3               0.00          54.49         2.92    0.03  11.52      21.50   
4               0.04          19.70         4.80   11.10   1.14      18.30   
5              22.57          17.34         6.45   35.57   0.29      14.25   
6              19.05          26.17        18.34    2.24   4.26      21.05   
7               1.93           6.97         4.16   61.55   4.84       6.98   
8               0.00          69.28      

fuzzy time! cmeans using xie-beni, FPC; doing the macro sweep across many m and many k

In [5]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from scipy.spatial.distance import cdist

# =============================================================================
# PATHS
# =============================================================================

csv_path = Path(r"C:\NCA_DATA\Vegetation Data\NCA_Master_Aligned_spp_fg.csv")
out_dir = Path(r"C:\NCA_DATA\Vegetation Data\cluster\fuzzy_veg")
out_dir.mkdir(parents=True, exist_ok=True)

# =============================================================================
# SETTINGS
# =============================================================================

K_MIN = 2
K_MAX = 25
RANDOM_STATE = 42
N_INIT = 20
MAX_ITER = 300
TOL = 1e-5

# FCM requires m > 1.0
M_LIST = [1.5, 2.0]

# =============================================================================
# LOAD DATA
# =============================================================================

print("Reading vegetation dataset...")
df = pd.read_csv(csv_path)
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

# =============================================================================
# IDENTIFY COLUMNS
# =============================================================================

meta_cols = {
    "Plot", "Year",
    "total_cover", "total_cover_fg",
    "SHRUB_total"
}

fg_cols = [c for c in df.columns if c.endswith("_FG")]

print(f"Detected {len(fg_cols)} FG columns.")

# =============================================================================
# BUILD FG MATRIX
# =============================================================================

X_raw = df[fg_cols].copy()

for c in fg_cols:
    X_raw[c] = pd.to_numeric(X_raw[c], errors="coerce")

X_raw = X_raw.fillna(0.0)

row_sums = X_raw.sum(axis=1).values
keep = row_sums > 0

if not np.all(keep):
    print(f"Dropping {(~keep).sum():,} rows with zero FG total.")

df_work = df.loc[keep].copy().reset_index(drop=True)
X_raw = X_raw.loc[keep].reset_index(drop=True)

# Hellinger transform
print("Applying Hellinger transform to FG matrix...")
row_sums = X_raw.sum(axis=1).values[:, None]
X_prop = X_raw.values / row_sums
X = np.sqrt(X_prop)

print(f"FG matrix shape: {X.shape}")
print("FG columns used:", fg_cols)

# =============================================================================
# HELPERS
# =============================================================================

def fit_kmeans(X: np.ndarray, k: int, random_state: int):
    km = KMeans(
        n_clusters=k,
        random_state=random_state,
        n_init=N_INIT,
        max_iter=1000
    )
    labels = km.fit_predict(X)
    return km, labels

def init_membership(n: int, k: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    U = rng.random((n, k))
    U = U / U.sum(axis=1, keepdims=True)
    return U

def fuzzy_cmeans(
    X: np.ndarray,
    k: int,
    m: float,
    max_iter: int = 300,
    tol: float = 1e-5,
    seed: int = 42
):
    """
    Basic fuzzy c-means.
    X: (n, p)
    Returns:
      centers: (k, p)
      U: (n, k)
      jm: objective history
    """
    if m <= 1.0:
        raise ValueError("FCM requires m > 1.0")

    n = X.shape[0]
    U = init_membership(n, k, seed)
    jm = []

    eps = 1e-12

    for _ in range(max_iter):
        U_old = U.copy()

        Um = U ** m
        centers = (Um.T @ X) / (Um.sum(axis=0)[:, None] + eps)

        D = cdist(X, centers, metric="euclidean")
        D = np.fmax(D, eps)

        # If any point is exactly at a center, give it full membership there
        zero_mask = D <= eps
        if np.any(zero_mask):
            U = np.zeros_like(D)
            row_idx = np.where(zero_mask.any(axis=1))[0]
            for i in row_idx:
                j = np.argmin(D[i])
                U[i, j] = 1.0
            nonzero_rows = np.setdiff1d(np.arange(n), row_idx)
            if len(nonzero_rows) > 0:
                Dnz = D[nonzero_rows]
                power = -2.0 / (m - 1.0)
                tmp = Dnz ** power
                U[nonzero_rows] = tmp / tmp.sum(axis=1, keepdims=True)
        else:
            power = -2.0 / (m - 1.0)
            tmp = D ** power
            U = tmp / tmp.sum(axis=1, keepdims=True)

        obj = np.sum((U ** m) * (D ** 2))
        jm.append(obj)

        if np.max(np.abs(U - U_old)) < tol:
            break

    return centers, U, np.array(jm)

def fuzzy_partition_coefficient(U: np.ndarray) -> float:
    n = U.shape[0]
    return np.sum(U ** 2) / n

def partition_entropy(U: np.ndarray) -> float:
    eps = 1e-12
    return -np.sum(U * np.log(U + eps)) / U.shape[0]

def xie_beni_index(X: np.ndarray, centers: np.ndarray, U: np.ndarray, m: float) -> float:
    eps = 1e-12
    D = cdist(X, centers, metric="euclidean")
    num = np.sum((U ** m) * (D ** 2))

    if centers.shape[0] < 2:
        return np.nan

    center_dist = cdist(centers, centers, metric="euclidean")
    np.fill_diagonal(center_dist, np.inf)
    min_sep_sq = np.min(center_dist) ** 2

    return num / (X.shape[0] * (min_sep_sq + eps))

def save_metric_plot(dfm: pd.DataFrame, metric: str, title: str, out_path: Path):
    plt.figure(figsize=(8, 5))
    for m in sorted(dfm["m"].unique()):
        sub = dfm[dfm["m"] == m].sort_values("k")
        plt.plot(sub["k"], sub[metric], marker="o", label=f"m={m}")
    plt.xlabel("k")
    plt.ylabel(metric)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

# =============================================================================
# HARD K-MEANS REFERENCE (m = 1 LIMIT)
# =============================================================================

print("\nRunning hard k-means reference...")
hard_results = []

for k in range(K_MIN, K_MAX + 1):
    km, labels = fit_kmeans(X, k, RANDOM_STATE)

    sil = silhouette_score(X, labels, metric="euclidean")
    ch = calinski_harabasz_score(X, labels)

    hard_results.append({
        "method": "kmeans",
        "m": 1.0,
        "k": k,
        "silhouette": float(sil),
        "calinski_harabasz": float(ch),
        "wcss": float(km.inertia_)
    })

hard_metrics = pd.DataFrame(hard_results)

print("\nTop hard k by silhouette:")
print(hard_metrics.sort_values("silhouette", ascending=False).head(10).to_string(index=False))

# =============================================================================
# FUZZY C-MEANS SWEEP
# =============================================================================

print("\nRunning fuzzy c-means diagnostics...")
fuzzy_results = []

for m in M_LIST:
    print(f"\n--- m = {m} ---")
    for k in range(K_MIN, K_MAX + 1):
        print(f"  k = {k}")
        centers, U, jm = fuzzy_cmeans(
            X, k, m,
            max_iter=MAX_ITER,
            tol=TOL,
            seed=RANDOM_STATE
        )

        hard_labels = np.argmax(U, axis=1)

        fpc = fuzzy_partition_coefficient(U)
        pe = partition_entropy(U)
        xb = xie_beni_index(X, centers, U, m)

        sil = silhouette_score(X, hard_labels, metric="euclidean")
        ch = calinski_harabasz_score(X, hard_labels)

        fuzzy_results.append({
            "method": "fcm",
            "m": m,
            "k": k,
            "fpc": float(fpc),
            "partition_entropy": float(pe),
            "xie_beni": float(xb),
            "silhouette_hardened": float(sil),
            "calinski_harabasz_hardened": float(ch),
            "objective_final": float(jm[-1]),
            "n_iter": int(len(jm))
        })

fuzzy_metrics = pd.DataFrame(fuzzy_results)

# =============================================================================
# PLOTS
# =============================================================================

save_metric_plot(
    fuzzy_metrics, "fpc",
    "Fuzzy vegetation clustering: FPC (higher better)",
    out_dir / "fcm_fpc.png"
)

save_metric_plot(
    fuzzy_metrics, "partition_entropy",
    "Fuzzy vegetation clustering: partition entropy (lower better)",
    out_dir / "fcm_partition_entropy.png"
)

save_metric_plot(
    fuzzy_metrics, "xie_beni",
    "Fuzzy vegetation clustering: Xie-Beni (lower better)",
    out_dir / "fcm_xie_beni.png"
)

save_metric_plot(
    fuzzy_metrics, "silhouette_hardened",
    "Fuzzy vegetation clustering: hardened silhouette",
    out_dir / "fcm_silhouette_hardened.png"
)

print("\nPlots written to:")
for fn in [
    "fcm_fpc.png",
    "fcm_partition_entropy.png",
    "fcm_xie_beni.png",
    "fcm_silhouette_hardened.png"
]:
    print(out_dir / fn)

# =============================================================================
# CONSOLE SUMMARY
# =============================================================================

print("\nTop FCM by FPC:")
print(
    fuzzy_metrics.sort_values(["fpc", "xie_beni"], ascending=[False, True])
    .head(12)
    .to_string(index=False)
)

print("\nTop FCM by lowest Xie-Beni:")
print(
    fuzzy_metrics.sort_values("xie_beni", ascending=True)
    .head(12)
    .to_string(index=False)
)

print("\nTop FCM by lowest partition entropy:")
print(
    fuzzy_metrics.sort_values("partition_entropy", ascending=True)
    .head(12)
    .to_string(index=False)
)

# =============================================================================
# PICK BEST MODEL FOR INSPECTION
# =============================================================================
# Simple rule:
#   - prioritize low Xie-Beni
#   - among those, prefer higher FPC
#   - and avoid extreme k if several are similar
# You can change this after inspecting the curves.

best_row = (
    fuzzy_metrics.sort_values(
        ["xie_beni", "fpc", "partition_entropy"],
        ascending=[True, False, True]
    )
    .iloc[0]
)

BEST_M = float(best_row["m"])
BEST_K = int(best_row["k"])

print(f"\nSelected exploratory best model: m = {BEST_M}, k = {BEST_K}")

centers, U, jm = fuzzy_cmeans(
    X, BEST_K, BEST_M,
    max_iter=MAX_ITER,
    tol=TOL,
    seed=RANDOM_STATE
)

hard_labels = np.argmax(U, axis=1) + 1
max_membership = np.max(U, axis=1)

df_inspect = df_work.copy()
df_inspect["veg_cluster_fuzzy"] = hard_labels
df_inspect["max_membership"] = max_membership

print("\nCluster sizes from hardened fuzzy labels:")
print(df_inspect["veg_cluster_fuzzy"].value_counts().sort_index().to_string())

if fg_cols:
    fg_summary = (
        df_inspect.groupby("veg_cluster_fuzzy")[fg_cols]
        .mean()
        .round(2)
    )

    print("\nFG summary for selected fuzzy model:")
    print(fg_summary.to_string())

print("\nMembership concentration summary:")
print(pd.Series(max_membership).describe().to_string())

Reading vegetation dataset...
Rows: 717
Columns: 85
Detected 10 FG columns.
Applying Hellinger transform to FG matrix...
FG matrix shape: (717, 10)
FG columns used: ['ARTR_FG', 'BAREGROUND_FG', 'BIOCRUST_FG', 'EAG_FG', 'EF_FG', 'LITTER_FG', 'NPF_FG', 'OTHER_FG', 'PBG_FG', 'SHRUB_FG']

Running hard k-means reference...


c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\c


Top hard k by silhouette:
method   m  k  silhouette  calinski_harabasz       wcss
kmeans 1.0  6    0.251602         164.842500 140.671911
kmeans 1.0  5    0.247248         173.884666 153.647658
kmeans 1.0  7    0.228527         153.089495 132.424094
kmeans 1.0  4    0.224257         176.593838 174.261332
kmeans 1.0  2    0.215579         221.150997 231.988481
kmeans 1.0 10    0.214237         131.711064 113.478309
kmeans 1.0  9    0.212817         136.501201 119.471610
kmeans 1.0  8    0.211327         143.541208 125.659494
kmeans 1.0  3    0.208153         184.459215 200.266704
kmeans 1.0 12    0.202277         122.732764 104.200741

Running fuzzy c-means diagnostics...

--- m = 1.5 ---
  k = 2
  k = 3
  k = 4
  k = 5
  k = 6
  k = 7
  k = 8
  k = 9
  k = 10
  k = 11
  k = 12
  k = 13
  k = 14
  k = 15
  k = 16
  k = 17
  k = 18
  k = 19
  k = 20
  k = 21
  k = 22
  k = 23
  k = 24
  k = 25

--- m = 2.0 ---
  k = 2
  k = 3
  k = 4
  k = 5
  k = 6
  k = 7
  k = 8
  k = 9
  k = 10
  k 

In [10]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

# ============================================================
# LOAD
# ============================================================

csv_path = r"C:\NCA_DATA\Vegetation Data\NCA_Master_Aligned_spp_fg.csv"
df = pd.read_csv(csv_path)

fg_cols = [c for c in df.columns if c.endswith("_FG")]

X_raw = df[fg_cols].copy()
X_raw = X_raw.fillna(0.0)

# remove zero rows
row_sums = X_raw.sum(axis=1).values
keep = row_sums > 0

X_raw = X_raw.loc[keep].reset_index(drop=True)

# Hellinger transform
X_prop = X_raw.values / X_raw.sum(axis=1).values[:, None]
X = np.sqrt(X_prop)

# ============================================================
# FCM
# ============================================================

def fuzzy_cmeans(X, k, m, max_iter=300, tol=1e-5, seed=42):
    n = X.shape[0]
    rng = np.random.default_rng(seed)

    U = rng.random((n, k))
    U = U / U.sum(axis=1, keepdims=True)

    eps = 1e-12

    for _ in range(max_iter):
        U_old = U.copy()

        Um = U ** m
        centers = (Um.T @ X) / (Um.sum(axis=0)[:, None] + eps)

        D = cdist(X, centers)
        D = np.fmax(D, eps)

        power = -2.0 / (m - 1.0)
        tmp = D ** power
        U = tmp / tmp.sum(axis=1, keepdims=True)

        if np.max(np.abs(U - U_old)) < tol:
            break

    return centers, U

# ============================================================
# METRICS
# ============================================================

def compute_metrics(U):
    n = U.shape[0]

    fpc = np.sum(U**2) / n
    max_membership = np.max(U, axis=1)

    return {
        "fpc": fpc,
        "membership_mean": np.mean(max_membership),
        "membership_std": np.std(max_membership),
        "membership_p25": np.percentile(max_membership, 25),
        "membership_p50": np.percentile(max_membership, 50),
        "membership_p75": np.percentile(max_membership, 75),
    }

# ============================================================
# SWEEP
# ============================================================

k_values = range(5, 11)
m_values = np.round(np.arange(1.3, 1.8, 0.1), 2)

results = []

for m in m_values:
    for k in k_values:
        centers, U = fuzzy_cmeans(X, k=k, m=m)

        metrics = compute_metrics(U)

        results.append({
            "k": k,
            "m": m,
            **metrics
        })

results_df = pd.DataFrame(results)

# ============================================================
# PRINT SUMMARY
# ============================================================

print("\n=== FULL SWEEP RESULTS ===\n")
print(results_df.sort_values(["m", "k"]).to_string(index=False))

print("\n=== BEST BY FPC (per m) ===\n")
print(
    results_df.loc[
        results_df.groupby("m")["fpc"].idxmax()
    ].sort_values("m").to_string(index=False)
)

print("\n=== BEST BY MEMBERSHIP MEAN (per m) ===\n")
print(
    results_df.loc[
        results_df.groupby("m")["membership_mean"].idxmax()
    ].sort_values("m").to_string(index=False)
)


=== FULL SWEEP RESULTS ===

 k   m      fpc  membership_mean  membership_std  membership_p25  membership_p50  membership_p75
 5 1.3 0.788281         0.858291        0.164586        0.778095        0.929220        0.986464
 6 1.3 0.756240         0.836484        0.173453        0.751948        0.904166        0.979865
 7 1.3 0.738011         0.820427        0.187452        0.690801        0.896781        0.978426
 8 1.3 0.721156         0.807056        0.192480        0.668979        0.882741        0.973961
 9 1.3 0.722069         0.809264        0.191983        0.673539        0.881861        0.975230
10 1.3 0.725007         0.811768        0.189918        0.690201        0.884899        0.974350
 5 1.4 0.677577         0.782436        0.181081        0.656437        0.826524        0.943231
 6 1.4 0.639515         0.748957        0.200367        0.586013        0.801409        0.931522
 7 1.4 0.621440         0.735871        0.203898        0.571281        0.777011        0.921141
 

Parent clustering for 7, k = 1.5
this was drawn as the most representative partitioning of our expected states from previous clustering: when we had POSE as an 'Other' it ordinated around that variable. as soon as we removed that and reclassified BASC as an EF, we got a clustering that mapped cleanly to our expectations of 'state endmembers' like intact sage, invaded shrub, eag-pbg transition, homog eag, etc.

In [13]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

# =============================================================================
# PATHS
# =============================================================================

csv_path = Path(r"C:\NCA_DATA\Vegetation Data\NCA_Master_Aligned_spp_fg.csv")

out_dir = Path(r"C:\NCA_DATA\Vegetation Data\cluster\fuzzy_veg")
out_dir.mkdir(parents=True, exist_ok=True)

assignments_out = out_dir / "veg_fuzzy_m1p5_k7_point_assignments.csv"

# =============================================================================
# SETTINGS
# =============================================================================

M = 1.5
K = 7
RANDOM_STATE = 42
MAX_ITER = 300
TOL = 1e-5

# =============================================================================
# LOAD DATA
# =============================================================================

df = pd.read_csv(csv_path)
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# =============================================================================
# BUILD FG MATRIX
# =============================================================================

meta_cols = {
    "Plot", "Year",
    "total_cover", "total_cover_fg",
    "SHRUB_total"
}

fg_cols = [c for c in df.columns if c.endswith("_FG")]

X_raw = df[fg_cols].copy()

for c in fg_cols:
    X_raw[c] = pd.to_numeric(X_raw[c], errors="coerce")

X_raw = X_raw.fillna(0.0)

row_sums = X_raw.sum(axis=1).values
keep = row_sums > 0

df_work = df.loc[keep].copy().reset_index(drop=True)
X_raw = X_raw.loc[keep].reset_index(drop=True)

# Hellinger transform
row_sums = X_raw.sum(axis=1).values[:, None]
X_prop = X_raw.values / row_sums
X = np.sqrt(X_prop)

# =============================================================================
# FUZZY C-MEANS
# =============================================================================

def init_membership(n: int, k: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    U = rng.random((n, k))
    return U / U.sum(axis=1, keepdims=True)


def fuzzy_cmeans(X, k, m, max_iter=300, tol=1e-5, seed=42):
    if m <= 1.0:
        raise ValueError("FCM requires m > 1.0")

    n = X.shape[0]
    U = init_membership(n, k, seed)
    jm = []
    eps = 1e-12

    for _ in range(max_iter):
        U_old = U.copy()

        Um = U ** m
        centers = (Um.T @ X) / (Um.sum(axis=0)[:, None] + eps)

        D = cdist(X, centers, metric="euclidean")
        D = np.fmax(D, eps)

        power = -2.0 / (m - 1.0)
        tmp = D ** power
        U = tmp / tmp.sum(axis=1, keepdims=True)

        obj = np.sum((U ** m) * (D ** 2))
        jm.append(obj)

        if np.max(np.abs(U - U_old)) < tol:
            break

    return centers, U, np.array(jm)


centers, U, jm = fuzzy_cmeans(
    X,
    K,
    M,
    max_iter=MAX_ITER,
    tol=TOL,
    seed=RANDOM_STATE
)

# =============================================================================
# ATTACH LABELS + MEMBERSHIPS
# =============================================================================

hard_labels = np.argmax(U, axis=1) + 1
max_membership = np.max(U, axis=1)

df_out = df_work.copy()
df_out["veg_cluster_fuzzy"] = hard_labels
df_out["max_membership"] = max_membership

for i in range(K):
    df_out[f"membership_{i+1}"] = U[:, i]

# =============================================================================
# WRITE OUTPUT
# =============================================================================

df_out.to_csv(assignments_out, index=False)

print(f"\nWrote point-level fuzzy cluster assignments:")
print(assignments_out)

print("\nCluster sizes:")
print(df_out["veg_cluster_fuzzy"].value_counts().sort_index().to_string())

print("\nMembership summary:")
print(df_out["max_membership"].describe().to_string())

print("\nFG summary:")
print(
    df_out
    .groupby("veg_cluster_fuzzy")[fg_cols]
    .mean()
    .round(2)
    .to_string()
)


Wrote point-level fuzzy cluster assignments:
C:\NCA_DATA\Vegetation Data\cluster\fuzzy_veg\veg_fuzzy_m1p5_k7_point_assignments.csv

Cluster sizes:
veg_cluster_fuzzy
1     85
2     96
3     90
4     87
5    127
6    115
7    117

Membership summary:
count    717.000000
mean       0.650410
std        0.203614
min        0.221973
25%        0.478758
50%        0.659867
75%        0.828474
max        0.989045

FG summary:
                   ARTR_FG  BAREGROUND_FG  BIOCRUST_FG  EAG_FG  EF_FG  LITTER_FG  NPF_FG  OTHER_FG  PBG_FG  SHRUB_FG
veg_cluster_fuzzy                                                                                                   
1                    25.18          22.13        14.93    8.99   1.48      14.24    0.00      1.09    9.19      2.78
2                     0.18          13.29         7.47   29.11   3.50      12.27    0.00      1.02   32.16      0.96
3                     0.30          21.10        16.70    3.01   1.64      17.12    0.00      0.97    8.40   

In [18]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# =============================================================================
# INPUTS
# =============================================================================

csv_path = Path(r"C:\NCA_DATA\Vegetation Data\NCA_Master_Aligned_spp_fg.csv")
out_dir = Path(r"C:\NCA_DATA\Vegetation Data\cluster\fuzzy_veg\cluster_diagnostics")
out_dir.mkdir(parents=True, exist_ok=True)

# If you already have df_inspect in memory from fuzzy c-means, use that instead.
# Otherwise, load your saved/working dataframe here.
# This assumes df_inspect includes:
#   veg_cluster_fuzzy
#   max_membership

#Uncomment if needed:
#df_inspect = pd.read_csv(csv_path)

CORE_MEMBERSHIP = 0.60
SUBCLUSTER_K = [2, 3, 4]
RANDOM_STATE = 42

# =============================================================================
# COLUMN SETUP
# =============================================================================

assign_path = Path(r"C:\NCA_DATA\Vegetation Data\cluster\fuzzy_veg\veg_fuzzy_m1p5_k7_point_assignments.csv")
print(df.columns)
df = pd.read_csv(assign_path)
df = df.drop(columns=["Unnamed: 0"], errors="ignore")
meta_cols = {
    "Plot", "Year",
    "total_cover", "total_cover_fg",
    "SHRUB_total",
    "veg_cluster_fuzzy",
    "max_membership",
    "X",
}

fg_cols = [c for c in df.columns if c.endswith("_FG")]
species_cols = [
    c for c in df.columns
    if c not in meta_cols
    and c not in fg_cols
    and not c.endswith("_total")
    and not c.startswith("membership_")
]

print(f"FG columns: {fg_cols}")
print(f"Species/catchall columns: {len(species_cols)}")

# =============================================================================
# HELPERS
# =============================================================================

def hellinger_matrix(df_sub, cols):
    X = df_sub[cols].copy()
    X = X.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    row_sums = X.sum(axis=1).values

    keep = row_sums > 0
    X = X.loc[keep].copy()
    row_sums = row_sums[keep]

    X_prop = X.values / row_sums[:, None]
    X_hell = np.sqrt(X_prop)

    return X_hell, X.index

def top_means(df_sub, cols, n=12):
    vals = (
        df_sub[cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
        .mean()
        .sort_values(ascending=False)
        .head(n)
    )
    return vals

def inspect_cluster(cluster_id):
    sub_all = df[df["veg_cluster_fuzzy"] == cluster_id].copy()
    sub_core = sub_all[sub_all["max_membership"] >= CORE_MEMBERSHIP].copy()

    print("\n" + "=" * 80)
    print(f"VEG FUZZY CLUSTER {cluster_id}")
    print("=" * 80)
    print(f"n total: {len(sub_all)}")
    print(f"n core (membership >= {CORE_MEMBERSHIP}): {len(sub_core)}")

    print("\nMembership summary:")
    print(sub_all["max_membership"].describe().to_string())

    print("\nFG mean composition:")
    fg_mean = top_means(sub_all, fg_cols, n=len(fg_cols))
    print(fg_mean.round(2).to_string())

    print("\nTop species/catchall means:")
    sp_mean = top_means(sub_all, species_cols, n=15)
    print(sp_mean.round(2).to_string())

    if len(sub_core) >= 10:
        print("\nTop CORE species/catchall means:")
        core_sp_mean = top_means(sub_core, species_cols, n=15)
        print(core_sp_mean.round(2).to_string())

    return sub_all, sub_core

# =============================================================================
# OVERALL CLUSTER SUMMARY
# =============================================================================

print("\nCluster sizes:")
print(df["veg_cluster_fuzzy"].value_counts().sort_index().to_string())

print("\nMean membership by cluster:")
print(
    df.groupby("veg_cluster_fuzzy")["max_membership"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(3)
    .to_string()
)

print("\nFG summary by fuzzy cluster:")
fg_summary = (
    df.groupby("veg_cluster_fuzzy")[fg_cols]
    .mean()
    .round(2)
)
print(fg_summary.to_string())

# =============================================================================
# INSPECT EACH PRIMARY CLUSTER
# =============================================================================

cluster_ids = sorted(df["veg_cluster_fuzzy"].dropna().unique())

cluster_outputs = {}

for cid in cluster_ids:
    sub_all, sub_core = inspect_cluster(cid)
    cluster_outputs[cid] = {"all": sub_all, "core": sub_core}

# =============================================================================
# OPTIONAL: SPECIES-LEVEL SUBCLUSTERING WITHIN EACH FUZZY CLUSTER
# =============================================================================

for cid in cluster_ids:
    sub = cluster_outputs[cid]["core"]

    if len(sub) < 20:
        print(f"\nSkipping subclustering for cluster {cid}: too few core samples ({len(sub)})")
        continue

    X_hell, kept_idx = hellinger_matrix(sub, species_cols)
    sub_kept = sub.loc[kept_idx].copy()

    if X_hell.shape[0] < 20:
        print(f"\nSkipping subclustering for cluster {cid}: too few nonzero rows")
        continue

    print("\n" + "-" * 80)
    print(f"SUBCLUSTERING PRIMARY CLUSTER {cid}")
    print("-" * 80)

    best = None

    for k in SUBCLUSTER_K:
        if X_hell.shape[0] <= k:
            continue

        km = KMeans(
            n_clusters=k,
            random_state=RANDOM_STATE,
            n_init=20,
            max_iter=1000
        )

        labels = km.fit_predict(X_hell)

        if len(set(labels)) > 1 and min(pd.Series(labels).value_counts()) >= 3:
            sil = silhouette_score(X_hell, labels)
        else:
            sil = np.nan

        print(f"k={k} | silhouette={sil:.3f}")

        if best is None or (not np.isnan(sil) and sil > best["silhouette"]):
            best = {
                "k": k,
                "labels": labels,
                "silhouette": sil
            }

    if best is None:
        continue

    sub_kept["species_subcluster"] = best["labels"] + 1

    print(f"\nBest subcluster model for cluster {cid}: k={best['k']}, silhouette={best['silhouette']:.3f}")

    print("\nSubcluster sizes:")
    print(sub_kept["species_subcluster"].value_counts().sort_index().to_string())

    print("\nSubcluster FG means:")
    print(
        sub_kept.groupby("species_subcluster")[fg_cols]
        .mean()
        .round(2)
        .to_string()
    )

    print("\nTop species per subcluster:")
    sp_summary = (
        sub_kept.groupby("species_subcluster")[species_cols]
        .mean()
    )

    for sid in sorted(sub_kept["species_subcluster"].unique()):
        print(f"\nPrimary cluster {cid} | species subcluster {sid}")
        print(
            sp_summary.loc[sid]
            .sort_values(ascending=False)
            .head(12)
            .round(2)
            .to_string()
        )

    # PCA plot for this primary cluster
    if X_hell.shape[0] >= 5:
        pca = PCA(n_components=2, random_state=RANDOM_STATE)
        X2 = pca.fit_transform(X_hell)

        plt.figure(figsize=(7, 5))
        plt.scatter(
            X2[:, 0],
            X2[:, 1],
            c=sub_kept["species_subcluster"],
            s=35,
            alpha=0.8,
            cmap="tab10"
        )
        plt.title(f"Primary fuzzy cluster {cid}: species subclusters")
        plt.xlabel("PC1")
        plt.ylabel("PC2")
        plt.tight_layout()
        plt.savefig(out_dir / f"cluster_{cid}_species_subclusters_pca.png", dpi=200)
        plt.close()

print("\nDone.")
print(f"Plots written to: {out_dir}")

Index(['X', 'Plot', 'Year', 'ARTR2', 'CHVI8', 'ERNA10', 'KRLA2', 'ATCO',
       'PUTR2', 'BAPR5', 'POSE', 'BRTE', 'AGCR', 'PSSP6', 'ACTH7', 'HECO',
       'ELEL5', 'ACHY', 'ELYMUS', 'GRSP', 'GUSA', 'SAVE4', 'SHRUB', 'BRBR',
       'PBG', 'ONAC', 'CHJU', 'FORB', 'BRASS', 'SATR12D', 'CETE5', 'LEPE2',
       'NOTR', 'UNK.PLANT', 'OTHER', 'LITTER', 'BAREGROUND', 'BASC', 'SATR12',
       'BIOCRUST', 'HEAN3', 'SIAL2D', 'LASE', 'SIAL2', 'CEST8', 'CHVI8D',
       'DESO2', 'VUOC', 'ARTR2D', 'AMAL', 'TEGL', 'BAPR5D', 'LECI4', 'EPBR3',
       'DEPI', 'TACA8', 'CREPI', 'FEID', 'EUMA7', 'ATCA2', 'SPGR2', 'PSJU3',
       'BAAM4', 'TRDU', 'MACA2', 'GAYOP', 'SPCR', 'TRMA3', 'ATCOD', 'VUBR',
       'TESP2', 'ATCA2D', 'BASA3', 'total_cover', 'ARTR_FG', 'BAREGROUND_FG',
       'BIOCRUST_FG', 'EAG_FG', 'EF_FG', 'LITTER_FG', 'NPF_FG', 'OTHER_FG',
       'PBG_FG', 'SHRUB_FG', 'SHRUB_total', 'total_cover_fg',
       'veg_cluster_fuzzy', 'max_membership', 'membership_1', 'membership_2',
       'membership_3',

c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(



--------------------------------------------------------------------------------
SUBCLUSTERING PRIMARY CLUSTER 2
--------------------------------------------------------------------------------
k=2 | silhouette=0.267
k=3 | silhouette=0.304


c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


k=4 | silhouette=0.296

Best subcluster model for cluster 2: k=3, silhouette=0.304

Subcluster sizes:
species_subcluster
1    27
2     6
3    18

Subcluster FG means:
                    ARTR_FG  BAREGROUND_FG  BIOCRUST_FG  EAG_FG  EF_FG  LITTER_FG  NPF_FG  OTHER_FG  PBG_FG  SHRUB_FG
species_subcluster                                                                                                   
1                      0.07          12.03         7.63   28.90   1.69      12.37     0.0      0.84   36.36      0.08
2                      0.14          14.44         0.89   24.55   2.78      18.63     0.0      0.07   38.15      0.27
3                      0.00          10.08         7.33   35.96   1.76       6.93     0.0      1.01   36.47      0.46

Top species per subcluster:

Primary cluster 2 | species subcluster 1
AGCR          32.40
BRTE          28.90
LITTER        12.31
BAREGROUND    12.03
BIOCRUST       7.63
POSE           2.89
OTHER          0.83
SATR12         0.75
PBG         

...

k=4 | silhouette=0.162

Best subcluster model for cluster 5: k=4, silhouette=0.162

Subcluster sizes:
species_subcluster
1    29
2    24
3    29
4     9

Subcluster FG means:
                    ARTR_FG  BAREGROUND_FG  BIOCRUST_FG  EAG_FG  EF_FG  LITTER_FG  NPF_FG  OTHER_FG  PBG_FG  SHRUB_FG
species_subcluster                                                                                                   
1                      1.82           5.75         4.92   76.42   2.44       4.08    0.00      1.24    2.66      0.65
2                      0.02           4.51         0.24   74.31   5.31      11.63    0.00      0.15    2.21      1.60
3                      1.18           2.58         0.10   86.43   1.36       0.99    0.01      0.62    5.05      1.67
4                      0.00           2.21         1.10   66.28  27.52       1.98    0.00      0.80    0.11      0.00

Top species per subcluster:

Primary cluster 5 | species subcluster 1
BRTE          76.42
BAREGROUND     5.75
BIOCRU

c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


k=3 | silhouette=0.337
k=4 | silhouette=0.370

Best subcluster model for cluster 6: k=4, silhouette=0.370

Subcluster sizes:
species_subcluster
1    16
2    11
3    35
4     9

Subcluster FG means:
                    ARTR_FG  BAREGROUND_FG  BIOCRUST_FG  EAG_FG  EF_FG  LITTER_FG  NPF_FG  OTHER_FG  PBG_FG  SHRUB_FG
species_subcluster                                                                                                   
1                       0.0          45.00         4.40    0.10  39.15       7.83    0.00      1.35    1.95      0.20
2                       0.0          32.16        10.07    2.10  43.49       7.93    0.00      0.47    3.77      0.00
3                       0.0          31.30        19.23    0.62  32.33      13.21    0.01      0.63    2.46      0.21
4                       0.0          31.71        10.22    1.04  34.96      20.64    0.00      0.40    0.81      0.18

Top species per subcluster:

Primary cluster 6 | species subcluster 1
BAREGROUND    45.00
CET

c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


k=2 | silhouette=0.216
k=3 | silhouette=0.275
k=4 | silhouette=0.307

Best subcluster model for cluster 7: k=4, silhouette=0.307

Subcluster sizes:
species_subcluster
1    16
2    24
3     8
4    13

Subcluster FG means:
                    ARTR_FG  BAREGROUND_FG  BIOCRUST_FG  EAG_FG  EF_FG  LITTER_FG  NPF_FG  OTHER_FG  PBG_FG  SHRUB_FG
species_subcluster                                                                                                   
1                       0.0          31.49        10.57    1.52   2.02       9.25     0.0      1.48   42.39      1.27
2                       0.4          30.09         9.22    1.16   4.18       5.98     0.0      0.51   47.45      1.00
3                       0.0          27.12        11.95    1.60   0.60       4.78     0.0      2.40   48.58      2.98
4                       0.0          26.94        11.88    2.21   0.96      12.59     0.0      1.29   43.70      0.37

Top species per subcluster:

Primary cluster 7 | species subcluster 1


Test how many subclusters fit per parent cluster:

In [19]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# =============================================================================
# PATHS
# =============================================================================

in_csv = Path(
    r"C:\NCA_DATA\Vegetation Data\cluster\fuzzy_veg\veg_fuzzy_m1p5_k7_point_assignments.csv"
)

out_dir = Path(r"C:\NCA_DATA\Vegetation Data\cluster\fuzzy_veg")
out_dir.mkdir(parents=True, exist_ok=True)

points_out = out_dir / "veg_fuzzy_m1p5_k7_with_species_subclusters.csv"
summary_out = out_dir / "veg_fuzzy_m1p5_k7_subcluster_summary.csv"

# =============================================================================
# SETTINGS
# =============================================================================

CORE_MEMBERSHIP = 0.60
SUBCLUSTER_K = [2, 3, 4]
RANDOM_STATE = 42

cluster_labels = {
    1: "ARTR_shrub_steppe",
    2: "PBG_EAG_transition",
    3: "non_ARTR_shrub",
    4: "EAG_shrub_mix",
    5: "EAG_extreme",
    6: "EF_extreme",
    7: "PBG_open",
}

# =============================================================================
# LOAD
# =============================================================================

df = pd.read_csv(in_csv)
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

fg_cols = [c for c in df.columns if c.endswith("_FG")]

exclude_cols = {
    "X",
    "Plot",
    "Year",
    "total_cover",
    "total_cover_fg",
    "SHRUB_total",
    "veg_cluster_fuzzy",
    "max_membership",
}

species_cols = [
    c for c in df.columns
    if c not in exclude_cols
    and c not in fg_cols
    and not c.startswith("membership_")
    and not c.endswith("_total")
]

print(f"Rows: {len(df)}")
print(f"FG columns: {len(fg_cols)}")
print(f"Species/catchall columns used for subclustering: {len(species_cols)}")

# =============================================================================
# HELPERS
# =============================================================================

def hellinger_matrix(df_sub, cols):
    X = (
        df_sub[cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
    )

    row_sums = X.sum(axis=1).values
    keep = row_sums > 0

    X = X.loc[keep].copy()
    row_sums = row_sums[keep]

    X_prop = X.values / row_sums[:, None]
    X_hell = np.sqrt(X_prop)

    return X_hell, X.index


def top_species_string(row, cols, n=5):
    vals = row[cols].sort_values(ascending=False).head(n)
    return "; ".join([f"{name}={value:.2f}" for name, value in vals.items()])


# =============================================================================
# RUN SUBCLUSTERING WITHIN EACH FUZZY CLUSTER
# =============================================================================

df["cluster_label"] = df["veg_cluster_fuzzy"].map(cluster_labels)
df["species_subcluster"] = np.nan
df["species_subcluster_label"] = np.nan
df["species_subcluster_silhouette"] = np.nan

subcluster_summary_rows = []

for cid in sorted(df["veg_cluster_fuzzy"].dropna().unique()):
    cid = int(cid)

    sub_all = df[df["veg_cluster_fuzzy"] == cid].copy()
    sub_core = sub_all[sub_all["max_membership"] >= CORE_MEMBERSHIP].copy()

    print("\n" + "=" * 80)
    print(f"PRIMARY CLUSTER {cid}: {cluster_labels.get(cid, 'unlabeled')}")
    print("=" * 80)
    print(f"n total: {len(sub_all)}")
    print(f"n core:  {len(sub_core)}")

    if len(sub_core) < 20:
        print("Skipping: too few core samples.")
        continue

    X_hell, kept_idx = hellinger_matrix(sub_core, species_cols)
    sub_kept = sub_core.loc[kept_idx].copy()

    if X_hell.shape[0] < 20:
        print("Skipping: too few nonzero species rows.")
        continue

    best = None

    for k in SUBCLUSTER_K:
        if X_hell.shape[0] <= k:
            continue

        km = KMeans(
            n_clusters=k,
            random_state=RANDOM_STATE,
            n_init=20,
            max_iter=1000
        )

        labels = km.fit_predict(X_hell)
        sizes = pd.Series(labels).value_counts()

        if len(set(labels)) > 1 and sizes.min() >= 3:
            sil = silhouette_score(X_hell, labels)
        else:
            sil = np.nan

        print(f"k={k} | silhouette={sil:.3f} | min_size={sizes.min()}")

        if best is None or (not np.isnan(sil) and sil > best["silhouette"]):
            best = {
                "k": k,
                "labels": labels,
                "silhouette": sil
            }

    if best is None:
        print("No valid subcluster model found.")
        continue

    sub_kept["species_subcluster"] = best["labels"] + 1

    print(
        f"Selected k={best['k']} | silhouette={best['silhouette']:.3f}"
    )

    # write labels back to full df for core points only
    for idx, label in zip(sub_kept.index, sub_kept["species_subcluster"]):
        df.loc[idx, "species_subcluster"] = int(label)
        df.loc[idx, "species_subcluster_label"] = f"{cid}.{int(label)}"
        df.loc[idx, "species_subcluster_silhouette"] = best["silhouette"]

    # summary rows
    grouped = sub_kept.groupby("species_subcluster")

    for sid, g in grouped:
        sid = int(sid)

        fg_mean = g[fg_cols].mean(numeric_only=True)
        sp_mean = g[species_cols].mean(numeric_only=True)

        row = {
            "veg_cluster_fuzzy": cid,
            "cluster_label": cluster_labels.get(cid, "unlabeled"),
            "species_subcluster": sid,
            "species_subcluster_label": f"{cid}.{sid}",
            "n": len(g),
            "parent_n_total": len(sub_all),
            "parent_n_core": len(sub_core),
            "core_membership_threshold": CORE_MEMBERSHIP,
            "selected_k": best["k"],
            "silhouette": best["silhouette"],
            "max_membership_mean": g["max_membership"].mean(),
            "max_membership_median": g["max_membership"].median(),
            "dominant_fg": fg_mean.idxmax().replace("_FG", ""),
            "dominant_fg_value": fg_mean.max(),
            "top_species": top_species_string(sp_mean, species_cols, n=5),
        }

        for c in fg_cols:
            row[c] = fg_mean[c]

        subcluster_summary_rows.append(row)

# =============================================================================
# BUILD SUMMARY TABLE
# =============================================================================

summary = pd.DataFrame(subcluster_summary_rows)

if not summary.empty:
    summary = summary.sort_values(
        ["veg_cluster_fuzzy", "species_subcluster"]
    )

# =============================================================================
# WRITE OUTPUTS
# =============================================================================

df.to_csv(points_out, index=False)
summary.to_csv(summary_out, index=False)

print("\nDone.")
print(f"Point-level output written:\n{points_out}")
print(f"Subcluster summary written:\n{summary_out}")

if not summary.empty:
    print("\nSubcluster summary:")
    show_cols = [
        "veg_cluster_fuzzy",
        "cluster_label",
        "species_subcluster",
        "n",
        "silhouette",
        "dominant_fg",
        "dominant_fg_value",
        "top_species",
    ]
    print(summary[show_cols].round(3).to_string(index=False))

Rows: 717
FG columns: 10
Species/catchall columns used for subclustering: 70

PRIMARY CLUSTER 1: ARTR_shrub_steppe
n total: 85
n core:  41
k=2 | silhouette=0.158 | min_size=18
k=3 | silhouette=0.163 | min_size=10
k=4 | silhouette=0.177 | min_size=8
Selected k=4 | silhouette=0.177

PRIMARY CLUSTER 2: PBG_EAG_transition
n total: 96
n core:  51
k=2 | silhouette=0.267 | min_size=24


c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\scottfordham\AppData\Local\Temp\ipykernel_17644\3508426787.py:17

k=3 | silhouette=0.304 | min_size=6
k=4 | silhouette=0.296 | min_size=7
Selected k=3 | silhouette=0.304

PRIMARY CLUSTER 3: non_ARTR_shrub
n total: 90
n core:  56
k=2 | silhouette=0.422 | min_size=20


c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


k=3 | silhouette=0.288 | min_size=14
k=4 | silhouette=0.280 | min_size=4
Selected k=2 | silhouette=0.422

PRIMARY CLUSTER 4: EAG_shrub_mix
n total: 87
n core:  44


...

k=3 | silhouette=0.337 | min_size=11
k=4 | silhouette=0.370 | min_size=9
Selected k=4 | silhouette=0.370

PRIMARY CLUSTER 7: PBG_open
n total: 117
n core:  61


c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


k=2 | silhouette=0.216 | min_size=26
k=3 | silhouette=0.275 | min_size=16


c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\scottfordham\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


k=4 | silhouette=0.307 | min_size=8
Selected k=4 | silhouette=0.307

Done.
Point-level output written:
C:\NCA_DATA\Vegetation Data\cluster\fuzzy_veg\veg_fuzzy_m1p5_k7_with_species_subclusters.csv
Subcluster summary written:
C:\NCA_DATA\Vegetation Data\cluster\fuzzy_veg\veg_fuzzy_m1p5_k7_subcluster_summary.csv

Subcluster summary:
 veg_cluster_fuzzy      cluster_label  species_subcluster  n  silhouette dominant_fg  dominant_fg_value                                                              top_species
                 1  ARTR_shrub_steppe                   1  8       0.177        ARTR             36.478 ARTR2=36.48; BAREGROUND=24.69; BIOCRUST=18.10; LITTER=6.36; SATR12D=3.57
                 1  ARTR_shrub_steppe                   2 13       0.177        ARTR             29.105    ARTR2=29.07; BAREGROUND=27.10; LITTER=22.57; POSE=7.25; BIOCRUST=5.64
                 1  ARTR_shrub_steppe                   3 10       0.177        ARTR             27.780   ARTR2=27.78; BIOCRUST=24.01; BA

Add subclustering logic and only keeping the ones that make sense (keep 2, 3, 6, 7) because 1 is an artr shrub state, 4 is eag separating on whether it is chvi or not, and 5 is EAG parttitioning around 60-80% cover with some mustards

In [20]:
from pathlib import Path
import numpy as np
import pandas as pd

# =============================================================================
# PATHS
# =============================================================================

in_csv = Path(
    r"C:\NCA_DATA\Vegetation Data\cluster\fuzzy_veg\veg_fuzzy_m1p5_k7_with_species_subclusters.csv"
)

out_dir = Path(r"C:\NCA_DATA\Vegetation Data\cluster\fuzzy_veg")
out_dir.mkdir(parents=True, exist_ok=True)

points_out = out_dir / "veg_fuzzy_m1p5_k7_kept_hierarchy_points.csv"
summary_out = out_dir / "veg_fuzzy_m1p5_k7_kept_hierarchy_summary.csv"

# =============================================================================
# SETTINGS
# =============================================================================

keep_subclusters_for = {2, 3, 6, 7}

cluster_labels = {
    1: "ARTR_shrub_steppe",
    2: "PBG_EAG_transition",
    3: "non_ARTR_shrub",
    4: "EAG_shrub_mix",
    5: "EAG_extreme",
    6: "EF_extreme",
    7: "PBG_open",
}

subcluster_labels = {
    "2.1": "AGCR_BRTE_transition",
    "2.2": "PBG_BRTE_litter_transition",
    "2.3": "BRTE_POSE_PSSP6_transition",

    "3.1": "CHVI8_shrub",
    "3.2": "KRLA2_ATCO_shrub",

    "6.1": "CETE5_SATR12_bareground_EF",
    "6.2": "BRASS_EF",
    "6.3": "BAPR5_biocrust_EF",
    "6.4": "LEPE2_litter_EF",

    "7.1": "AGCR_open_PBG",
    "7.2": "POSE_open_PBG",
    "7.3": "PSSP6_ELEL5_open_PBG",
    "7.4": "mixed_PBG_ACTH7_ELYMUS",
}

# =============================================================================
# LOAD
# =============================================================================

df = pd.read_csv(in_csv)
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

fg_cols = [c for c in df.columns if c.endswith("_FG")]

exclude_cols = {
    "X",
    "Plot",
    "Year",
    "total_cover",
    "total_cover_fg",
    "SHRUB_total",
    "veg_cluster_fuzzy",
    "max_membership",
    "cluster_label",
    "species_subcluster",
    "species_subcluster_label",
    "species_subcluster_silhouette",
}

species_cols = [
    c for c in df.columns
    if c not in exclude_cols
    and c not in fg_cols
    and not c.startswith("membership_")
    and not c.endswith("_total")
]

# =============================================================================
# BUILD KEPT HIERARCHY LABELS
# =============================================================================

df["cluster_label"] = df["veg_cluster_fuzzy"].map(cluster_labels)

df["species_subcluster"] = pd.to_numeric(
    df["species_subcluster"],
    errors="coerce"
)

# Keep subclusters only for selected parent clusters.
df["kept_subcluster"] = np.where(
    df["veg_cluster_fuzzy"].isin(keep_subclusters_for),
    df["species_subcluster"],
    np.nan
)

df["hierarchy_id"] = np.where(
    df["veg_cluster_fuzzy"].isin(keep_subclusters_for)
    & df["species_subcluster"].notna(),
    df["veg_cluster_fuzzy"].astype(int).astype(str)
    + "."
    + df["species_subcluster"].astype("Int64").astype(str),
    df["veg_cluster_fuzzy"].astype(int).astype(str)
)

df["hierarchy_label"] = df["hierarchy_id"].map(subcluster_labels)

df["hierarchy_label"] = df["hierarchy_label"].fillna(
    df["veg_cluster_fuzzy"].map(cluster_labels)
)

df["hierarchy_level"] = np.where(
    df["veg_cluster_fuzzy"].isin(keep_subclusters_for)
    & df["species_subcluster"].notna(),
    "subcluster",
    "parent"
)

# =============================================================================
# SUMMARY HELPERS
# =============================================================================

def top_species_string(vals, n=5):
    vals = vals.sort_values(ascending=False).head(n)
    return "; ".join([f"{k}={v:.2f}" for k, v in vals.items()])

def summarize_group(g):
    fg_mean = g[fg_cols].mean(numeric_only=True)
    sp_mean = g[species_cols].mean(numeric_only=True)

    row = {
        "n": len(g),
        "max_membership_mean": g["max_membership"].mean(),
        "max_membership_median": g["max_membership"].median(),
        "dominant_fg": fg_mean.idxmax().replace("_FG", ""),
        "dominant_fg_value": fg_mean.max(),
        "top_species": top_species_string(sp_mean, n=5),
    }

    for c in fg_cols:
        row[c] = fg_mean[c]

    return pd.Series(row)

# =============================================================================
# BUILD SUMMARY
# =============================================================================

summary = (
    df.groupby(
        [
            "hierarchy_id",
            "hierarchy_label",
            "hierarchy_level",
            "veg_cluster_fuzzy",
            "cluster_label",
        ],
        dropna=False
    )
    .apply(summarize_group, include_groups=False)
    .reset_index()
)

# Sort numeric hierarchy IDs like 1, 2.1, 2.2, ...
summary["_sort_parent"] = summary["hierarchy_id"].str.split(".").str[0].astype(int)
summary["_sort_child"] = (
    summary["hierarchy_id"]
    .str.split(".")
    .str[1]
    .fillna("0")
    .astype(int)
)

summary = (
    summary
    .sort_values(["_sort_parent", "_sort_child"])
    .drop(columns=["_sort_parent", "_sort_child"])
)

# =============================================================================
# WRITE OUTPUTS
# =============================================================================

df.to_csv(points_out, index=False)
summary.to_csv(summary_out, index=False)

print("\nWrote:")
print(points_out)
print(summary_out)

print("\n=== KEPT HIERARCHY SUMMARY ===")
show_cols = [
    "hierarchy_id",
    "hierarchy_label",
    "hierarchy_level",
    "n",
    "max_membership_mean",
    "dominant_fg",
    "dominant_fg_value",
    "top_species",
]

print(
    summary[show_cols]
    .round(3)
    .to_string(index=False)
)


Wrote:
C:\NCA_DATA\Vegetation Data\cluster\fuzzy_veg\veg_fuzzy_m1p5_k7_kept_hierarchy_points.csv
C:\NCA_DATA\Vegetation Data\cluster\fuzzy_veg\veg_fuzzy_m1p5_k7_kept_hierarchy_summary.csv

=== KEPT HIERARCHY SUMMARY ===
hierarchy_id            hierarchy_label hierarchy_level   n  max_membership_mean dominant_fg  dominant_fg_value                                                             top_species
           1          ARTR_shrub_steppe          parent  85                0.600        ARTR             25.180  ARTR2=25.05; BAREGROUND=22.13; BIOCRUST=14.93; LITTER=11.88; BRTE=8.99
           2         PBG_EAG_transition          parent  45                0.471         PBG             27.122    BRTE=26.82; BAREGROUND=15.17; LITTER=13.32; AGCR=8.55; BIOCRUST=8.31
         2.1       AGCR_BRTE_transition      subcluster  27                0.781         PBG             36.363   AGCR=32.40; BRTE=28.90; LITTER=12.31; BAREGROUND=12.03; BIOCRUST=7.63
         2.2 PBG_BRTE_litter_transition    